In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import datetime
import locale
locale.setlocale(locale.LC_TIME, 'ru_RU.UTF-8')
from io import BytesIO

In [2]:
# Словарь месяцев вручную, чтобы избежать проблем с кодировкой
months_ru = {
    1: "янв", 2: "фев", 3: "мар", 4: "апр", 5: "май", 6: "июн",
    7: "июл", 8: "авг", 9: "сен", 10: "окт", 11: "ноя", 12: "дек"
}

In [6]:
df = pd.read_excel(r"C:\Users\Mi\Downloads\052026_Продажи.xlsx", sheet_name = 'рейтинг')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Рейтинг              212 non-null    int64 
 1   Застройщик           212 non-null    object
 2   2026-05-01 00:00:00  212 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 5.1+ KB


In [7]:
df2

,Застройщик,Регион,Название ЖК,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,...,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,"Среднее за 2025 г., шт./мес",2026-01-01 00:00:00,2026-02-01 00:00:00,2026-03-01 00:00:00,2026-04-01 00:00:00,"Среднее за 20256г., шт./мес"
0,ПИК,Total,NaN,3034.0,4401.0,3756.0,3091.0,1616.0,1545.0,1861.0,...,1624.0,1820.0,1806.0,1460.0,2298.750000,1268.0,924.0,1211.0,1198.0,1150.25
1,ПИК,Москва,Total,1325.0,1893.0,1536.0,1442.0,887.0,938.0,1088.0,...,899.0,948.0,1055.0,801.0,1146.833333,597.0,449.0,643.0,648.0,584.25
2,ПИК,Москва,Бусиновский парк,128.0,208.0,146.0,133.0,104.0,102.0,79.0,...,43.0,43.0,30.0,26.0,94.250000,16.0,21.0,25.0,22.0,21.00
3,ПИК,Москва,Зеленый парк,142.0,137.0,79.0,95.0,80.0,43.0,63.0,...,68.0,73.0,51.0,57.0,78.333333,37.0,29.0,58.0,93.0,54.25
4,ПИК,Москва,Москворечье,106.0,203.0,225.0,155.0,58.0,54.0,35.0,...,27.0,23.0,56.0,43.0,85.166667,37.0,24.0,44.0,24.0,32.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983,Бэсткон,Москва,Время,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN
984,ЦентрСтрой,Total,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN
985,ЦентрСтрой,Новая Москва,Total,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN
986,ЦентрСтрой,Новая Москва,Эдельвейс (Рогозинино),1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN


In [5]:
df

,Рейтинг,Дата регистрации (месяц),2026-05-01 00:00:00
0,1,ПИК,1157
1,2,Самолет,583
2,3,Гранель,474
3,4,ДСК-1 (ФСК Лидер),362
4,5,ЛСР,226
...,...,...,...
207,208,Engeo Development,0
208,209,Hutton Development,0
209,210,Sawatsky,0
210,211,Бэсткон,0


In [8]:
developers = df["Застройщик"].tolist()

In [9]:
df2 = pd.read_excel(r"C:\Users\Mi\Downloads\052026_Продажи.xlsx", sheet_name = 'массив')

In [10]:
def format_col(col):
    if isinstance(col, (pd.Timestamp, datetime.datetime)):
        return f"{months_ru[col.month]}.{str(col.year)[-2:]}"
    return col

df2.columns = [format_col(col) for col in df2.columns]


In [11]:
# Фильтруем по каждому региону
def remove_total_if_one_jk(group):
    # Проверяем сколько строк с ЖК (не Total)
    jk_count = group[group["Название ЖК"] != "Total"].shape[0]
    if jk_count <= 1:
        # Удаляем строки с Total
        group = group[group["Название ЖК"] != "Total"]
    return group

In [12]:
# Проверяем результат
print(df2.columns.tolist())

['Застройщик', 'Регион', 'Название ЖК', 'янв.25', 'фев.25', 'мар.25', 'апр.25', 'май.25', 'июн.25', 'июл.25', 'авг.25', 'сен.25', 'окт.25', 'ноя.25', 'дек.25', 'Среднее за 2025 г., шт./мес', 'янв.26', 'фев.26', 'мар.26', 'апр.26', 'май.26', 'Среднее за 2026 г., шт./мес']


In [13]:
region_order = ['Total', 'Москва', 'Новая Москва', 'Московская область']

In [14]:
# 2. Создаём словарь: застройщик → порядковый номер
developer_numbers = {name: i + 1 for i, name in enumerate(df["Застройщик"].dropna().unique())}

# 3. Добавляем новый столбец с номерами
df2["№"] = df2["Застройщик"].map(developer_numbers)

# 4. Ставим столбец "№" в начало
df2 = df2[["№"] + [col for col in df2.columns if col != "№"]]





In [15]:
df2['Застройщик'] = pd.Categorical(df2['Застройщик'], categories=developers, ordered=True)
df2['Регион'] = pd.Categorical(df2['Регион'], categories=region_order, ordered=True)
df_sorted = df2.sort_values(['Застройщик', 'Регион']).reset_index(drop=True)

# 5. Дублируем шапку перед каждым застройщиком
header = pd.DataFrame([df_sorted.columns], columns=df_sorted.columns)  # создаём строку-шапку
result = pd.DataFrame(columns=df_sorted.columns)

for dev in df_sorted["№"].unique():
    block = df_sorted[df_sorted["№"] == dev]
    result = pd.concat([result, header, block], ignore_index=True)

df_sorted = result

# 5. Заменяем все значения "Total" на "Итого"
# df_sorted = df_sorted.replace("Total", "Итого")

df_cleaned = df_sorted.groupby("Регион", group_keys=False).apply(remove_total_if_one_jk)

C:\Users\Mi\AppData\Local\Temp\ipykernel_19840\2155248353.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = df_sorted.groupby("Регион", group_keys=False).apply(remove_total_if_one_jk)


In [16]:
df_sorted = df_cleaned

In [18]:
import numpy as np

# --- 2. Сохраняем в Excel ---
output_file = r"C:\Users\Mi\OneDrive\Desktop\Портянка_06_26.xlsx"
df_sorted.to_excel(output_file, index=False)

# приводим месяцы к числам
df_sorted['апр.26'] = pd.to_numeric(df_sorted['апр.26'], errors='coerce')
df_sorted['май.26'] = pd.to_numeric(df_sorted['май.26'], errors='coerce')

# расчет динамики
df_sorted['Динамика мес/мес,%'] = np.where(
    df_sorted['Регион'].str.strip().str.lower() == 'total',
    (df_sorted['май.26'] / df_sorted['апр.26'] - 1),
    None
)

# # округление до десятых процента
# df_sorted['Динамика мес/мес,%'] = df_sorted['Динамика мес/мес,%'].round(3)


# --- 3. Открываем для форматирования ---
wb = load_workbook(output_file)
ws = wb.active

# Эта часть удаляет строки Total там, где в регионе всего один проект

region_start = None
region_end = None
current_region = None
rows_to_delete = []

for row in range(2, ws.max_row + 1):  # первая строка — заголовок
    region = ws[f"C{row}"].value  # теперь регион в колонке C
    jk_name = ws[f"D{row}"].value  # Total / ЖК в колонке D

    if region is None:
        continue
    if jk_name is None:
        jk_name = ""

    if region != current_region:
        if region_start is not None:
            # Собираем все названия ЖК в регионе
            jk_list = [str(ws[f"D{r}"].value).strip() for r in range(region_start, region_end + 1)]
            total_count = sum(1 for name in jk_list if name.lower() == "total")
            real_jk_count = sum(1 for name in jk_list if name.lower() != "total")

            print(f"\nРегион: {current_region}")
            print(f"Список ЖК (включая Total): {jk_list}")
            print(f"Количество ЖК без Total: {real_jk_count}, Total: {total_count}")

            # Если ЖК всего 1, удаляем Total
            if real_jk_count == 1 and total_count > 0:
                for r in range(region_start, region_end + 1):
                    if str(ws[f"D{r}"].value).strip().lower() == "total":
                        rows_to_delete.append(r)
                        print(f"Удаляю строку {r} с 'Total'")

        current_region = region
        region_start = row

    region_end = row

# Проверяем последний регион
if region_start is not None:
    jk_list = [str(ws[f"D{r}"].value).strip() for r in range(region_start, region_end + 1)]
    total_count = sum(1 for name in jk_list if name.lower() == "total")
    real_jk_count = sum(1 for name in jk_list if name.lower() != "total")

    print(f"\nРегион: {current_region}")
    print(f"Список ЖК (включая Total): {jk_list}")
    print(f"Количество ЖК без Total: {real_jk_count}, Total: {total_count}")

    if real_jk_count == 1 and total_count > 0:
        for r in range(region_start, region_end + 1):
            if str(ws[f"D{r}"].value).strip().lower() == "total":
                rows_to_delete.append(r)
                print(f"Удаляю строку {r} с 'Total'")

# Удаляем строки с конца
for r in sorted(rows_to_delete, reverse=True):
    ws.delete_rows(r)

# Эта часть добавляет шапку для каждого застройщика, а также добавляет серую заливку и жирный шрифт

# Форматы
fill = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid")
bold_font = Font(bold=True)
center_align = Alignment(vertical="center", horizontal="center")

# --- 4. Форматируем строки-шапки ---
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    if row[0].value == "№":  # шапка начинается со "№"
        for cell in row:
            cell.font = bold_font
            cell.fill = fill
            cell.alignment = center_align

# --- 5. Объединяем одинаковые подряд ячейки ---
def merge_identical_cells(column_idx):
    start = 2  # пропускаем первую строку
    current_value = ws[f"{get_column_letter(column_idx)}{start}"].value
    for row in range(3, ws.max_row + 2):
        cell_value = ws[f"{get_column_letter(column_idx)}{row}"].value
        if cell_value != current_value:
            if row - start > 1 and current_value is not None:
                ws.merge_cells(start_row=start, start_column=column_idx,
                               end_row=row - 1, end_column=column_idx)
                ws[f"{get_column_letter(column_idx)}{start}"].alignment = center_align
            start = row
            current_value = cell_value

# Объединяем по нужным столбцам
merge_identical_cells(1)  # №
merge_identical_cells(2)  # Застройщик
merge_identical_cells(3)  # Регион
merge_identical_cells(4)  # Название ЖК (если нужно)




# --- 6. Сохраняем итог ---
wb.save(output_file)
print("✅ Готово! Шапки добавлены, выделены цветом, и ячейки объединены.")



Регион: Регион
Список ЖК (включая Total): ['Название ЖК']
Количество ЖК без Total: 1, Total: 0

Регион: Total
Список ЖК (включая Total): ['None']
Количество ЖК без Total: 1, Total: 0

Регион: Москва
Список ЖК (включая Total): ['Total', 'Бусиновский парк', 'Зеленый парк', 'Москворечье', 'Люблинский парк', 'Кавказский 51', 'Мичуринский парк', 'Матвеевский парк', 'Алтуфьевское 53', 'Второй Иртышский', 'Волжский парк', 'Амурский Парк', 'Полар', 'Холланд Парк', 'Большая Академическая 85', 'Плеханова 11', 'Никольские луга', 'Квартал Мит', 'Митинский лес', 'Руставели 14', 'Нарвин', 'Первый Дубровский', 'Кронштадтский 9', 'Онежский вал', 'Барклая 6', 'Сигнальный 16', 'Строгино 360', 'Кутузовский квартал', 'Лосиноостровский парк', 'Второй Нагатинский', 'Кольская 8', 'Римского-Корсакова 11']
Количество ЖК без Total: 31, Total: 1

Регион: Новая Москва
Список ЖК (включая Total): ['Total', 'Саларьево Парк', 'Середневский лес', 'Бунинская набережная', 'Юнино']
Количество ЖК без Total: 4, Total: 1



In [ ]:
# пока не нужно
# df['Динамика мес/мес,%'] = np.where(
#     df['Регион'] == 'Total',
#     (df['фев.26'] / df['янв.26'] - 1) * 100,
#     None
# )

In [19]:
df_result = pd.read_excel(r"C:\Users\Mi\OneDrive\Desktop\Портянка_06_26.xlsx")

In [22]:
df_result.head()

,№,Застройщик,Регион,Название ЖК,янв.25,фев.25,мар.25,апр.25,май.25,июн.25,...,ноя.25,дек.25,"Среднее за 2025 г., шт./мес",янв.26,фев.26,мар.26,апр.26,май.26,"Среднее за 2026 г., шт./мес","Динамика мес/мес,%"
0,№,Застройщик,Регион,Название ЖК,янв.25,фев.25,мар.25,апр.25,май.25,июн.25,...,ноя.25,дек.25,"Среднее за 2025 г., шт./мес",янв.26,фев.26,NaN,NaN,май.26,"Среднее за 2026 г., шт./мес",NaN
1,1,ПИК,Итого,Название ЖК,3026,4394,3754,3088,1615,1543,...,1805,1460,2296.25,1267,924,1211.0,1196.0,1157,1151,-0.012386
2,1,ПИК,Москва,Итого,1320,1889,1534,1439,886,936,...,1054,801,1144.833333,596,449,643.0,646.0,562,579.2,-0.012386
3,1,ПИК,Москва,Бусиновский парк,128,208,146,133,104,102,...,30,26,94.25,16,21,25.0,22.0,20,20.8,-0.012386
4,1,ПИК,Москва,Зеленый парк,138,134,78,93,79,41,...,51,57,76.833333,37,29,58.0,91.0,55,54,-0.012386


In [21]:
df_result = df_result.ffill()

In [23]:
is_total = df_result['Название ЖК'].astype(str).str.contains('Итого', na=False)

In [24]:
base_df = df_result[~is_total]

grouped = base_df.groupby('Застройщик').agg({
    'Название ЖК': 'nunique',
    'Регион': 'nunique'
}).rename(columns={
    'Название ЖК': 'jk_count',
    'Регион': 'region_count'
})

In [25]:
df_result = df_result.merge(grouped, on='Застройщик', how='left')

In [26]:
to_delete = (
    is_total &
    (
        (df_result['jk_count'] == 1) |
        (df_result['region_count'] == 1)
    )
)

In [27]:
df_result = df_result[~to_delete]

In [28]:
df_result = df_result.drop(columns=['jk_count', 'region_count'])

In [29]:
output_file = r"C:\Users\Mi\OneDrive\Desktop\Портянка_06_26-2.xlsx"
df_result.to_excel(output_file, index=False)